In [1]:
import os
import cv2
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from pathlib import Path
import shutil

In [2]:
ORIGINAL_DATASET_DIR = '/kaggle/input/datasets/almiraraisa/aptos-2019'
ORIGINAL_IMAGES      = f'{ORIGINAL_DATASET_DIR}/train_images'
ORIGINAL_CSV         = f'{ORIGINAL_DATASET_DIR}/train.csv'
OUTPUT_DIR           = '/kaggle/working/aptos_2019_splitted'
OUTPUT_IMAGES        = f'{OUTPUT_DIR}/images'
IMG_SIZE             = 224   # change to 300 for EfficientNet-B3
os.makedirs(OUTPUT_IMAGES, exist_ok=True)

CLASS_NAMES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']

In [3]:
df = pd.read_csv(ORIGINAL_CSV)
df['id_code'] = df['id_code'].astype(str).apply(
    lambda x: x if x.endswith('.png') else x + '.png')

In [4]:
train_df, temp_df = train_test_split(
    df, test_size=0.20, stratify=df['diagnosis'], random_state=55)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['diagnosis'], random_state=55)

In [5]:
def print_split_stats(dataframe, split_name):
    total = len(dataframe)
    print(f"\n{'─'*55}")
    print(f"  {split_name} Split — {total} images")
    print(f"{'─'*55}")
    counts = dataframe['diagnosis'].value_counts().sort_index()
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        n   = counts.get(cls_idx, 0)
        pct = 100 * n / total
        bar = '█' * int(20 * n / total)
        print(f"  Class {cls_idx} ({cls_name:<20s}): "
              f"{n:4d}  ({pct:5.2f}%)  {bar}")

print_split_stats(df,       "Original Dataset")
print_split_stats(train_df, "Train (80%)")
print_split_stats(val_df,   "Validation (10%)")
print_split_stats(test_df,  "Test (10%)")

expected = {
    'train': round(len(df) * 0.80),
    'val':   round(len(df) * 0.10),
    'test':  round(len(df) * 0.10),
}
print(f"\n  Expected  → Train: ~{expected['train']}, "
      f"Val: ~{expected['val']}, Test: ~{expected['test']}")
print(f"  Actual    → Train:  {len(train_df)}, "
      f"Val:  {len(val_df)}, Test:  {len(test_df)}")


───────────────────────────────────────────────────────
  Original Dataset Split — 3662 images
───────────────────────────────────────────────────────
  Class 0 (No DR               ): 1805  (49.29%)  █████████
  Class 1 (Mild                ):  370  (10.10%)  ██
  Class 2 (Moderate            ):  999  (27.28%)  █████
  Class 3 (Severe              ):  193  ( 5.27%)  █
  Class 4 (Proliferative DR    ):  295  ( 8.06%)  █

───────────────────────────────────────────────────────
  Train (80%) Split — 2929 images
───────────────────────────────────────────────────────
  Class 0 (No DR               ): 1444  (49.30%)  █████████
  Class 1 (Mild                ):  296  (10.11%)  ██
  Class 2 (Moderate            ):  799  (27.28%)  █████
  Class 3 (Severe              ):  154  ( 5.26%)  █
  Class 4 (Proliferative DR    ):  236  ( 8.06%)  █

───────────────────────────────────────────────────────
  Validation (10%) Split — 366 images
───────────────────────────────────────────────────────
  Cl

In [6]:
train_df.to_csv(f'{OUTPUT_DIR}/train_split.csv', index=False)
val_df.to_csv(  f'{OUTPUT_DIR}/val_split.csv',   index=False)
test_df.to_csv( f'{OUTPUT_DIR}/test_split.csv',  index=False)
print(f"\n  CSVs saved to {OUTPUT_DIR}")


  CSVs saved to /kaggle/working/aptos_2019_splitted


In [ ]:
def crop_black_border(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(
        thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        c = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(c)
        if w > img.shape[1] * 0.1 and h > img.shape[0] * 0.1:
            img = img[y:y+h, x:x+w]
    return img

def ben_graham(img, size):
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    img = cv2.addWeighted(
        img, 4,
        cv2.GaussianBlur(img, (0, 0), size / 30), -4,
        128
    )
    return img

def preprocess(image_path, save_path, size=224):
    img = cv2.imread(str(image_path))
    if img is None:
        print(f"  [WARN] Cannot read: {image_path}")
        return False
    img = crop_black_border(img)
    img = ben_graham(img, size)
    cv2.imwrite(str(save_path), img)
    return True

In [ ]:
failed = 0
for img_name in tqdm(df['id_code'].values, desc="Preprocessing"):
    src = Path(ORIGINAL_IMAGES) / img_name
    dst = Path(OUTPUT_IMAGES)   / img_name
    if not dst.exists():
        ok = preprocess(src, dst, size=IMG_SIZE)
        if not ok:
            failed += 1
print(f"  Done. Failed: {failed}/{len(df)}")

Preprocessing: 100%|██████████| 3662/3662 [09:58<00:00,  6.12it/s]

  Done. Failed: 0/3662


In [9]:
zip_path = f'/kaggle/working/aptos_2019_{IMG_SIZE}px'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print(f"\n  ZIP: {zip_path}.zip")


  ZIP: /kaggle/working/aptos_2019_224px.zip
